# Notebook 1 — Exploration et prise en main de Spark

## Q1 — Création de la SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from time import perf_counter
import pandas as pd

spark = (
    SparkSession.builder
    .appName("TradeCorp ETL")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Version de Spark :", spark.version)
print("Nom de l'application :", spark.sparkContext.appName)

Version de Spark : 4.2.0
Nom de l'application : TradeCorp ETL


## Q2 — Lecture des huit fichiers CSV

In [2]:
DATA_PATH = "/home/jovyan/data"

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{DATA_PATH}/{nom_fichier}")
    )

df_customers = lire_csv("customers.csv")
df_orders = lire_csv("orders.csv")
df_order_details = lire_csv("order_details.csv")
df_products = lire_csv("products.csv")
df_categories = lire_csv("categories.csv")
df_suppliers = lire_csv("suppliers.csv")
df_employees = lire_csv("employees.csv")
df_shippers = lire_csv("shippers.csv")

dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

print("Nombre de DataFrames chargés :", len(dataframes))

Nombre de DataFrames chargés : 8


## Q3 — Exploration des schémas

In [3]:
for nom, df in dataframes.items():
    print(f"\n===== SCHÉMA : {nom} =====")
    df.printSchema()


===== SCHÉMA : customers =====
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)


===== SCHÉMA : orders =====
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (

## Q4 — Affichage des cinq premières lignes

In [4]:
for nom, df in dataframes.items():
    print(f"\n===== DONNÉES : {nom} =====")
    df.show(5, truncate=False)


===== DONNÉES : customers =====
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|company_name                      |contact_name      |contact_title       |address                      |city       |region|postal_code|country|phone         |fax           |
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|ALFKI      |Alfreds Futterkiste               |Maria Anders      |Sales Representative|Obere Str. 57                |Berlin     |NULL  |12209      |Germany|030-0074321   |030-0076545   |
|ANATR      |Ana Trujillo Emparedados y helados|Ana Trujillo      |Owner               |Avda. de la Constitución 2222|México D.F.|NULL  |05021      |Mexico |(5) 555-4729  |(5) 555-3745  |
|ANTON      |Antonio Moreno

## Q5 — Nombre de lignes par DataFrame

In [5]:
comptages = []

for nom, df in dataframes.items():
    nombre = df.count()
    comptages.append((nom, nombre))

df_comptages = spark.createDataFrame(
    comptages,
    ["table", "nombre_lignes"]
)

df_comptages.orderBy("table").show(truncate=False)

+-------------+-------------+
|table        |nombre_lignes|
+-------------+-------------+
|categories   |8            |
|customers    |91           |
|employees    |9            |
|order_details|2155         |
|orders       |830          |
|products     |77           |
|shippers     |6            |
|suppliers    |29           |
+-------------+-------------+



### Observation

Les nombres de lignes correspondent au brief, sauf pour `shippers.csv`.
Le brief annonce 3 transporteurs, mais le fichier fourni contient réellement
6 lignes. Je conserve les données du fichier source sans les modifier.

## Q6 — Statistiques descriptives

In [6]:
df_orders.select("freight").summary(
    "min",
    "max",
    "mean",
    "stddev"
).show()

+-------+------------------+
|summary|           freight|
+-------+------------------+
|    min|              0.02|
|    max|           1007.64|
|   mean| 78.24420481927719|
| stddev|116.77929363024192|
+-------+------------------+



In [7]:
df_products.select(
    "unit_price",
    "units_in_stock",
    "units_on_order",
    "reorder_level"
).summary(
    "min",
    "max",
    "mean",
    "stddev"
).show()

+-------+------------------+------------------+------------------+------------------+
|summary|        unit_price|    units_in_stock|    units_on_order|     reorder_level|
+-------+------------------+------------------+------------------+------------------+
|    min|               2.5|                 0|                 0|                 0|
|    max|             263.5|               125|               100|                30|
|   mean|28.833896103896105|40.506493506493506| 10.12987012987013|12.467532467532468|
| stddev| 33.82931122234885| 36.14722213124929|23.141072309089104| 10.93110532233639|
+-------+------------------+------------------+------------------+------------------+



## Q7 — Lazy evaluation

Spark utilise la **lazy evaluation**, ou évaluation paresseuse. Lorsqu'une
transformation est écrite, Spark ne traite pas immédiatement les données.
Il construit d'abord un plan d'exécution logique.

Le calcul est réellement déclenché lorsqu'une **action** est exécutée.

### Transformations

Une transformation crée un nouveau DataFrame sans lancer immédiatement le
calcul.

Exemples :

- `select()`
- `filter()`
- `withColumn()`

### Actions

Une action déclenche l'exécution du plan Spark et produit un résultat.

Exemples :

- `count()`
- `show()`
- `collect()`

Cette approche permet à Spark d'optimiser le plan d'exécution avant de
parcourir les données.

## Q8 — Consultation de Spark UI

In [8]:
(
    df_orders
    .groupBy("ship_country")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+------------+-----+
|ship_country|count|
+------------+-----+
|Germany     |122  |
|USA         |122  |
|Brazil      |83   |
|France      |77   |
|UK          |56   |
|Venezuela   |46   |
|Austria     |40   |
|Sweden      |37   |
|Canada      |30   |
|Italy       |28   |
|Mexico      |28   |
|Spain       |23   |
|Finland     |22   |
|Belgium     |19   |
|Ireland     |19   |
|Denmark     |18   |
|Switzerland |18   |
|Argentina   |16   |
|Portugal    |13   |
|Poland      |7    |
+------------+-----+
only showing top 20 rows


### Observation Spark UI

- Un **job** correspond à un traitement déclenché par une action.
- Un **stage** représente un ensemble d'opérations pouvant être exécutées
  ensemble.
- Une **task** est une unité de travail exécutée sur une partition de données.

## Q9 — Comparaison Spark et Pandas

In [9]:
fichier_orders = f"{DATA_PATH}/orders.csv"

debut_pandas = perf_counter()

pandas_orders = pd.read_csv(fichier_orders)
nombre_pandas = len(pandas_orders)

temps_pandas = perf_counter() - debut_pandas


debut_spark = perf_counter()

spark_orders_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(fichier_orders)
)

# count() est indispensable car Spark utilise la lazy evaluation.
nombre_spark = spark_orders_test.count()

temps_spark = perf_counter() - debut_spark


print("Nombre de lignes Pandas :", nombre_pandas)
print("Temps Pandas :", round(temps_pandas, 6), "seconde(s)")
print("Nombre de lignes Spark :", nombre_spark)
print("Temps Spark :", round(temps_spark, 6), "seconde(s)")

Nombre de lignes Pandas : 830
Temps Pandas : 2.036437 seconde(s)
Nombre de lignes Spark : 830
Temps Spark : 3.882204 seconde(s)


### Comparaison

Sur ce petit fichier de 830 lignes, Pandas est généralement plus rapide,
car Spark doit initialiser son moteur et construire un plan d'exécution.

Pandas convient aux données de petite ou moyenne taille qui tiennent dans la
mémoire d'une seule machine.

Spark devient plus pertinent lorsque les données sont très volumineuses,
doivent être distribuées sur plusieurs machines ou nécessitent un pipeline
parallélisé et tolérant aux pannes.

## Q10 — Liste des colonnes et types

In [10]:
print("Colonnes de df_orders :")

for colonne in df_orders.columns:
    print("-", colonne)

Colonnes de df_orders :
- order_id
- customer_id
- employee_id
- order_date
- required_date
- shipped_date
- ship_via
- freight
- ship_name
- ship_address
- ship_city
- ship_region
- ship_postal_code
- ship_country


In [11]:
print("Types de df_orders :")

for nom_colonne, type_colonne in df_orders.dtypes:
    print(f"{nom_colonne} : {type_colonne}")

Types de df_orders :
order_id : int
customer_id : string
employee_id : int
order_date : date
required_date : date
shipped_date : date
ship_via : int
freight : double
ship_name : string
ship_address : string
ship_city : string
ship_region : string
ship_postal_code : string
ship_country : string


### Colonnes qui nécessitent un cast

- `order_date` doit être convertie en date.
- `required_date` doit être convertie en date.
- `shipped_date` doit être convertie en date.
- `order_id`, `employee_id` et `ship_via` doivent être vérifiés comme entiers.
- `freight` doit être vérifiée comme valeur numérique de type double.

Les conversions seront réalisées dans le notebook de nettoyage.